In [1]:
import sys
from pathlib import Path

# Permite importar lista4 desde la carpeta src
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

from lista4.data import load_csv
from lista4.models import logistic_summary

df = load_csv("clientes")
objetivo = "abandona_30d"
predictoras = [c for c in df.columns if c != objetivo]

print(df.shape, "| proporción de abandono:", round(df[objetivo].mean(), 4))
df.head()

(1200, 7) | proporción de abandono: 0.2992


,antiguedad_meses,uso_30d_previos,facturas_vencidas_180d,tickets_90d,pago_automatico,ajuste_comercial_30d,abandona_30d
0,66,37.927153,0,1,0,23.039078,0
1,12,46.656223,0,1,1,19.158849,0
2,49,22.990662,0,3,1,27.712278,0
3,49,24.546478,1,0,0,96.997980,1
4,30,64.413222,0,2,1,0.000000,0


In [2]:
#A
variables_disponibles = [
    "antiguedad_meses",
    "uso_30d_previos",
    "facturas_vencidas_180d",
    "tickets_90d",
    "pago_automatico",
]

variables_disponibles

['antiguedad_meses',
 'uso_30d_previos',
 'facturas_vencidas_180d',
 'tickets_90d',
 'pago_automatico']

In [3]:
#B
m1 = logistic_summary(df, predictoras, objetivo, label="M1 (todas las variables)")
m2 = logistic_summary(df, variables_disponibles, objetivo, label="M2 (solo disponibles en t0)")

print("M1 convergió:", m1["converge"])
print("M2 convergió:", m2["converge"])

M1 convergió: True
M2 convergió: True


In [4]:
#C
metricas = pd.DataFrame([
    {k: v for k, v in m.items() if not k.startswith("_")}
    for m in (m1, m2)
]).set_index("modelo")

metricas[["riesgo_logistico", "errores", "prop_errores"]]

,riesgo_logistico,errores,prop_errores
modelo,,,
M1 (todas las variables),0.342485,160,0.133333
M2 (solo disponibles en t0),0.571716,336,0.280000


### (d)

| Modelo | Variables | Riesgo empírico logístico | Errores de clasificación | Proporción de errores |
|---|---|---|---|---|
| M1 (todas) | 6 | 0.3425 | 160 | 0.1333 (13.3 %) |
| M2 (solo disponibles en t0) | 5 | 0.5717 | 336 | 0.2800 (28.0 %) |

M1 tiene un riesgo menor y menos de la mitad de los errores que M2 (160 frente a 336), con un total de 1200 clientes. La única diferencia entre los dos modelos es la variable `ajuste_comercial_30d`.

### (e) ¿Qué modelo puede usarse en t0?

Si el modelo debe ejecutarse exactamente en t0, solo puede usarse **M2**, porque utiliza únicamente variables que ya se conocen en ese momento.

La variable que produce la diferencia entre M1 y M2 es **`ajuste_comercial_30d`**: se construye con datos entre t0 y t0 + 30, es decir, con información del futuro que todavía no existe cuando hay que hacer la predicción.

M1 parece mejor (riesgo 0.3425 y 13.3 % de errores, frente a 0.5717 y 28.0 % de M2), pero esa ventaja no es real: en producción esa variable no estaría disponible, así que M1 no podría reproducir ese desempeño. El desempeño que se puede esperar de verdad en t0 es el de M2.